In [1]:
!pip install -r ../requirements.txt 

In [1]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from threading import Thread
from pathlib import Path
import uuid
import shutil
import nest_asyncio
import uvicorn
import traceback
import subprocess
import time
import gc

nest_asyncio.apply()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def process_video(video_path: Path, use_rembg: bool = True, use_downsample: bool = False) -> Path:
    from PIL import Image
    import numpy as np
    from plyfile import PlyData
    from io import BytesIO

    run_uuid = str(uuid.uuid4())
    run_dir = Path("../data") / run_uuid
    frames_dir = run_dir / "frames"
    masked_dir = run_dir / "masked_frames"
    colmap_dir = run_dir / "colmap"
    brush_dir = run_dir / "brush"
    run_dir.mkdir(parents=True, exist_ok=True)
    frames_dir.mkdir()
    if use_rembg:
        masked_dir.mkdir()
    colmap_dir.mkdir()
    brush_dir.mkdir()

    input_video_path = run_dir / "input.mp4"
    shutil.copy(video_path, input_video_path)

    subprocess.run([
        "ffmpeg", "-hwaccel", "cuda", "-i", str(input_video_path),
        "-vf", "fps=2", str(frames_dir / "%04d.jpg")
    ], check=True)

    if use_downsample:
        downsample_dir = run_dir / "frames_2"
        downsample_dir.mkdir(exist_ok=True)
        subprocess.run([
            "magick", "mogrify", "-path", str(downsample_dir),
            "-resize", "50%", str(frames_dir / "*.jpg")
        ])
        frames_dir = downsample_dir

    if use_rembg:
        import onnxruntime as ort
        from torchvision import transforms
        import torch

        onnx_model_path = "/home/jovyan/datafabric/birefnetgeneral-model/birefnet-general.onnx"
        onnx_session = ort.InferenceSession(onnx_model_path, providers=["CUDAExecutionProvider"])
        input_name = onnx_session.get_inputs()[0].name
        onnx_transform = transforms.Compose([
            transforms.Resize((1024, 1024)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -250, 250)))
        for img_file in frames_dir.glob("*.jpg"):
            img = Image.open(img_file).convert("RGB")
            w, h = img.size
            input_tensor = onnx_transform(img).unsqueeze(0).numpy().astype(np.float32)
            pred = onnx_session.run(None, {input_name: input_tensor})[0].squeeze()
            mask = Image.fromarray((sigmoid(pred) * 255).astype(np.uint8)).resize((w, h))
            mask_arr = np.array(mask) / 255.0
            result_img = img.copy()
            result_img.putalpha(Image.fromarray((mask_arr * 255).astype(np.uint8)))
            result_img.save(masked_dir / f"{img_file.stem}.png")
        frames_dir = masked_dir
        del onnx_session, input_tensor, pred, result_img, mask_arr, mask
        torch.cuda.empty_cache()
        gc.collect()
        print(" ONNX session cleared and GPU memory released.")

    DB_PATH = colmap_dir / "database.db"
    subprocess.run([
        "colmap", "feature_extractor",
        "--database_path", str(DB_PATH),
        "--image_path", str(frames_dir),
        "--ImageReader.single_camera", "1",
        "--ImageReader.camera_model", "PINHOLE",
        "--SiftExtraction.use_gpu", "1"
    ], check=True)

    subprocess.run([
        "colmap", "exhaustive_matcher",
        "--database_path", str(DB_PATH),
        "--SiftMatching.use_gpu", "1"
    ], check=True)

    SPARSE_PATH = colmap_dir / "sparse"
    SPARSE_PATH.mkdir(exist_ok=True)

    subprocess.run([
        "colmap", "mapper",
        "--database_path", str(DB_PATH),
        "--image_path", str(frames_dir),
        "--output_path", str(SPARSE_PATH)
    ], check=True)

    image_output_dir = SPARSE_PATH / "0"
    image_output_dir.mkdir(parents=True, exist_ok=True)
    for img in frames_dir.glob("*.[jp][pn]g"):
        shutil.copy(img, image_output_dir)

    BRUSH_BIN = "/home/jovyan/brush/target/release/brush_app"
    export_name = f"{run_uuid}.ply"
    colmap_model_path = SPARSE_PATH / "0"
    subprocess.run([
        BRUSH_BIN, str(colmap_model_path),
        "--total-steps", "30000",
        "--export-path", str(brush_dir),
        "--export-name", export_name,
        "--export-every", "30000"
    ], check=True)

    def process_ply_to_splat(ply_file_path):
        plydata = PlyData.read(ply_file_path)
        vert = plydata["vertex"]
        sorted_indices = np.argsort(
            -np.exp(vert["scale_0"] + vert["scale_1"] + vert["scale_2"])
            / (1 + np.exp(-vert["opacity"]))
        )
        buffer = BytesIO()
        for idx in sorted_indices:
            v = vert[idx]
            position = np.array([v["x"], v["y"], v["z"]], dtype=np.float32)
            scales = np.exp(np.array([v["scale_0"], v["scale_1"], v["scale_2"]], dtype=np.float32))
            rot = np.array([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], dtype=np.float32)
            SH_C0 = 0.28209479177387814
            color = np.array([
                0.5 + SH_C0 * v["f_dc_0"],
                0.5 + SH_C0 * v["f_dc_1"],
                0.5 + SH_C0 * v["f_dc_2"],
                1 / (1 + np.exp(-v["opacity"])),
            ])
            buffer.write(position.tobytes())
            buffer.write(scales.tobytes())
            buffer.write((color * 255).clip(0, 255).astype(np.uint8).tobytes())
            buffer.write(((rot / np.linalg.norm(rot)) * 128 + 128).clip(0, 255).astype(np.uint8).tobytes())
        return buffer.getvalue()

    def save_splat_file(splat_data, output_path):
        with open(output_path, "wb") as f:
            f.write(splat_data)

    ply_file_path = brush_dir / export_name
    splat_file_path = ply_file_path.with_suffix(".splat")
    splat_data = process_ply_to_splat(ply_file_path)
    save_splat_file(splat_data, splat_file_path)

    return splat_file_path

@app.post("/upload/")
async def upload_video(file: UploadFile = File(...), use_rembg: bool = Form(True), use_downsample: bool = Form(False)):
    temp_dir = Path("uploads")
    temp_dir.mkdir(exist_ok=True)
    temp_file = temp_dir / f"{uuid.uuid4()}{Path(file.filename).suffix}"

    try:
        with open(temp_file, "wb") as f:
            shutil.copyfileobj(file.file, f)

        output_path = process_video(temp_file, use_rembg, use_downsample)

        return FileResponse(
            path=output_path,
            filename=output_path.name,
            media_type="application/octet-stream"
        )

    except subprocess.CalledProcessError as e:
        print(" Subprocess Error:", e)
        return JSONResponse(content={"error": "Subprocess failed", "details": str(e)}, status_code=500)

    except Exception as e:
        print(" Exception:", e)
        print(traceback.format_exc())
        return JSONResponse(content={"error": str(e), "trace": traceback.format_exc()}, status_code=500)

    finally:
        if temp_file.exists():
            temp_file.unlink()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

try:
    ip = subprocess.check_output("hostname -I", shell=True).decode().split()[0]
except Exception:
    ip = "localhost"

Thread(target=run, daemon=True).start()
print(f" FastAPI server is running at http://{ip}:8000/docs")
print(f" your endpoint to paste in UI is http://{ip}:8000/upload/")



 FastAPI server is running at http://10.137.137.6:8000/docs
 your endpoint to paste in UI is http://10.137.137.6:8000/upload/


INFO:     Started server process [44836]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable

 ONNX session cleared and GPU memory released.


I0619 17:57:37.340698 49365 misc.cc:44] 
Feature extraction
I0619 17:57:37.341634 49378 sift.cc:721] Creating SIFT GPU feature extractor
I0619 17:57:37.756603 49379 feature_extraction.cc:259] Processed file [1/88]
I0619 17:57:37.756669 49379 feature_extraction.cc:262]   Name:            0001.png
I0619 17:57:37.756673 49379 feature_extraction.cc:271]   Dimensions:      2160 x 3840
I0619 17:57:37.756676 49379 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0619 17:57:37.756681 49379 feature_extraction.cc:277]   Focal Length:    4608.00px
I0619 17:57:37.756697 49379 feature_extraction.cc:281]   Features:        8497
I0619 17:57:38.903700 49379 feature_extraction.cc:259] Processed file [2/88]
I0619 17:57:38.903735 49379 feature_extraction.cc:262]   Name:            0002.png
I0619 17:57:38.903738 49379 feature_extraction.cc:271]   Dimensions:      2160 x 3840
I0619 17:57:38.903740 49379 feature_extraction.cc:274]   Camera:          #1 - PINHOLE
I0619 17:57:38.903746 49379 featur

INFO:     10.137.137.1:54786 - "POST /upload/ HTTP/1.1" 200 OK


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab